In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
import xgboost as xgb
from sklearn import linear_model
from tqdm import tqdm

In [3]:
# ==========================
# GÉNÉRATION DES DATES DE TRAIN/TEST
# ==========================

path = 'data/'

X_train = pd.read_csv(path + 'X_train.csv',index_col='ROW_ID')
y_train = pd.read_csv(path + 'y_train.csv',index_col='ROW_ID')

def FAR(row):
    ret = row[RET_features].values
    vol = row[SIGNED_VOLUME_features].values
    return np.sum(ret * vol) / np.sum(np.abs(vol))

RET_features = [f'RET_{i}' for i in range(1,20)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,20)]
TURNOVER_features = ['AVG_DAILY_TURNOVER']
for i in [3,5,10,15,20]:
    X_train[ f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(1)
    X_train[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
    
X_train['FAR'] = X_train.apply(FAR, axis=1)
features = RET_features + SIGNED_VOLUME_features + TURNOVER_features + ['FAR']
features = features + [ f'AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]
features = features + [ f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]

In [ ]:
# exemple minimal
from keras import layers, models
input_shape = (65, 2, 20)
model = models.Sequential()
model.add(layers.Input(shape=input_shape))
model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(65*2, activation='sigmoid'))
model.add(layers.Reshape((65, 2, 20)))
model.compile(optimizer='adam', loss='mse')


In [ ]:
x_noisy = X_train[RET_features] + np.random.normal(0, 0.1, X_train[RET_features].shape)
x_clean = X_train[RET_features]
